In [13]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import logging
import os

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/shazizisazi/Datasets/a954a8e0efef2124456be0c386f4cb7979b928ad/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [3]:
import logging

logging.basicConfig(level=logging.INFO)

def lemmatization(text):
    try:
        lemmatizer = WordNetLemmatizer()
        return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
    except Exception as e:
        logging.warning(f"Lemmatization failed: {e}")
        return text

def remove_stopwords(text):
    try:
        stop_words = set(stopwords.words('english'))
        return ' '.join([word for word in text.split() if word not in stop_words])
    except Exception as e:
        logging.warning(f"Stopword removal failed: {e}")
        return text

def remove_numbers(text):
    try:
        return ''.join([i for i in text if not i.isdigit()])
    except Exception as e:
        logging.warning(f"Number removal failed: {e}")
        return text

def lower_case(text):
    try:
        return ' '.join([word.lower() for word in text.split()])
    except Exception as e:
        logging.warning(f"Lowercase conversion failed: {e}")
        return text

def remove_punctuation(text):
    try:
        return text.translate(str.maketrans('', '', string.punctuation))
    except Exception as e:
        logging.warning(f"Punctuation removal failed: {e}")
        return text

def remove_urls(text):
    try:
        return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    except Exception as e:
        logging.warning(f"URL removal failed: {e}")
        return text

def remove_small_sentences(text):
    try:
        return ' '.join([word for word in text.split() if len(word) > 2])
    except Exception as e:
        logging.warning(f"Small word removal failed: {e}")
        return text

In [4]:
def normalize_text(df):
    try:
        if 'content' not in df.columns:
            raise ValueError("Column 'content' not found in dataframe")

        df['content'] = df['content'].astype(str)

        df['content'] = df['content'].apply(lemmatization)
        df['content'] = df['content'].apply(remove_stopwords)
        df['content'] = df['content'].apply(remove_numbers)
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_punctuation)
        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(remove_small_sentences)

        logging.info("Text normalization completed successfully")
        return df
    
    except Exception as e:
        logging.error(f"Error in text normalization: {e}")
        raise

In [5]:
normalize_text(df)
df.head()

INFO:root:Text normalization completed successfully


,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [6]:
x=df['sentiment'].isin(['happiness','sadness'])
df=df[x]

In [7]:
df.head()

,sentiment,content
1,sadness,layin bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
6,sadness,sleep not thinking old friend want married now...
8,sadness,charviray charlene love miss
9,sadness,kelcouch sorry least friday


In [8]:
df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})
df.sample(5)

C:\Users\pc\AppData\Local\Temp\ipykernel_10696\2069291767.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})


,sentiment,content
19430,0,rainbowanne well nice thought lasted maybe lea...
30409,1,top cashier work today woot woot drink
35653,1,missing mother days happy mothers day
26596,1,azmomofmanyhats rocknrod gailelaine sarahstanl...
26172,1,todays day twitterlandhouse closing begin get ...


In [9]:
vectorizer=CountVectorizer(max_features=500)
x=vectorizer.fit_transform(df['content'])
y=df['sentiment']

In [10]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)

In [11]:
import mlflow

import dagshub
mlflow.set_tracking_uri('https://dagshub.com/shivamtiwari83032-collab/mini-mlops-proj.mlflow')
dagshub.init(repo_owner='shivamtiwari83032-collab', repo_name='mini-mlops-proj', mlflow=True)
mlflow.set_experiment('logistic_regression baseline')
    

INFO:httpx:HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Accessing as shivamtiwari83032-collab

INFO:dagshub:Accessing as shivamtiwari83032-collab
INFO:httpx:HTTP Request: GET https://dagshub.com/api/v1/repos/shivamtiwari83032-collab/mini-mlops-proj "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Initialized MLflow to track repo "shivamtiwari83032-collab/mini-mlops-proj"

INFO:dagshub:Initialized MLflow to track repo "shivamtiwari83032-collab/mini-mlops-proj"


Repository shivamtiwari83032-collab/mini-mlops-proj initialized!

INFO:dagshub:Repository shivamtiwari83032-collab/mini-mlops-proj initialized!


<Experiment: artifact_location='mlflow-artifacts:/dff517e7d37b46e684b6975e98b98733', creation_time=1776750775358, experiment_id='1', last_update_time=1776750775358, lifecycle_stage='active', name='logistic_regression baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [17]:
with mlflow.start_run():
    
    # Log preprocessing parameters
    mlflow.log_param("vectorizer", "Bag of Words")
    mlflow.log_param("num_features", 1000)
    mlflow.log_param("test_size", 0.2)
    
    # Model building and training
    model = LogisticRegression()
    model.fit(xtrain, ytrain)
    
    # Log model parameters
    mlflow.log_param("model", "Logistic Regression")
    
    # Model evaluation
    y_pred = model.predict(xtest)
    accuracy = accuracy_score(ytest, y_pred)
    precision = precision_score(ytest, y_pred)
    recall = recall_score(ytest, y_pred)
    f1 = f1_score(ytest, y_pred)
    
    # Log evaluation metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    # Log model
    mlflow.sklearn.log_model(model, "model")

    # Save and log the notebook

    
    # Print the results for verification
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    
    print(f"F1 Score: {f1}")

2026/04/21 15:18:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 15:18:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.7604819277108433
Precision: 0.7554240631163708
Recall: 0.7546798029556651
F1 Score: 0.7550517496303598
🏃 View run peaceful-ray-143 at: https://dagshub.com/shivamtiwari83032-collab/mini-mlops-proj.mlflow/#/experiments/1/runs/0d00b030c70e42bd981fb79f20055492
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mini-mlops-proj.mlflow/#/experiments/1
